# 06 Read Scaffold Results

Goal:

1. read final metrics from `results.json`
2. read log files line by line and only extract evaluate entries
3. combine them into two tables:
   - final metrics
   - convergence metrics

Prefer reading the experiment-to-output/log mapping from the manifest.


In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

CV_ROOT = Path('~/CV_Project').expanduser()
MANIFEST_PATH = CV_ROOT / 'output' / 'part1' / 'Re10k-1' / 'verification' / 'scaffold_40k' / 'manifest.json'

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
experiments = manifest['experiments']
print('loaded manifest from', MANIFEST_PATH)
print('num experiments =', len(experiments))


## 1. Read final results.json


In [ ]:
def flatten_dict(d, prefix=''):
    out = {}
    for k, v in d.items():
        key = f'{prefix}.{k}' if prefix else k
        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        else:
            out[key] = v
    return out

final_rows = []
for exp in experiments:
    result_path = Path(exp['output_dir']) / 'results.json'
    row = {
        'experiment_name': exp['experiment_name'],
        'scene': exp['dataset'],
        'plan': exp['plan'],
        'gs_model': exp['gs_model'],
        'output_dir': exp['output_dir'],
        'result_exists': result_path.exists(),
    }
    if result_path.exists():
        payload = json.loads(result_path.read_text(encoding='utf-8'))
        row.update(flatten_dict(payload))
    final_rows.append(row)

final_df = pd.DataFrame(final_rows)
final_df.head()


## 2. Read logs line by line and only keep evaluate entries


In [ ]:
pattern = re.compile(r'\[ITER\s+(\d+)\]\s+Evaluating\s+(test|train):\s+L1\s+([0-9eE.+-]+)\s+PSNR\s+([0-9eE.+-]+)')

conv_rows = []
for exp in experiments:
    log_path = Path(exp['log_file'])
    if not log_path.exists():
        continue
    with log_path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if not m:
                continue
            conv_rows.append({
                'experiment_name': exp['experiment_name'],
                'scene': exp['dataset'],
                'plan': exp['plan'],
                'gs_model': exp['gs_model'],
                'iter': int(m.group(1)),
                'split': m.group(2),
                'l1': float(m.group(3)),
                'psnr': float(m.group(4)),
                'log_file': str(log_path),
            })

conv_df = pd.DataFrame(conv_rows)
conv_df.head()


## 3. Quick check


In [ ]:
print('final_df shape =', final_df.shape)
print('conv_df shape =', conv_df.shape)

if not conv_df.empty:
    display(conv_df.groupby(['experiment_name', 'split'])['iter'].agg(['min', 'max', 'count']).reset_index())
else:
    print('No convergence rows parsed yet.')


## 4. Optional export


In [ ]:
EXPORT_DIR = MANIFEST_PATH.parent
final_csv = EXPORT_DIR / 'final_metrics.csv'
conv_csv = EXPORT_DIR / 'convergence_metrics.csv'

# Uncomment to export:
# final_df.to_csv(final_csv, index=False)
# conv_df.to_csv(conv_csv, index=False)

print('final csv path =', final_csv)
print('convergence csv path =', conv_csv)
